<a href="https://colab.research.google.com/github/HasanKhatib/iot-playground/blob/main/spark_ml_lib_with_pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Project with Mllib Pipeline

This guide will help you build a logistic regression model to predict Titanic survival using PySpark.

1. **Setup Environment**: Install necessary dependencies and set up the environment for Spark.
2. **Initialize Spark Session**: Configure and initialize a Spark session.
3. **Load Data**: Load and preprocess the Titanic dataset.
4. **Data Preprocessing**: Handle missing values and prepare the data for modeling.
5. **Feature Engineering**: Convert categorical variables to numerical format and assemble features.
6. **Build Pipeline**: Create a machine learning pipeline with stages for indexing, assembling features, and logistic regression.
7. **Train Model**: Fit the pipeline model on the training data.
8. **Make Predictions**: Use the trained model to make predictions on the test data.
9. **Evaluate Model**: Evaluate the model's performance using accuracy metric.

Follow the code cells below to implement each step in detail.


# Solution

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [2]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.3.0/spark-3.3.0-bin-hadoop3.tgz
!tar xf spark-3.3.0-bin-hadoop3.tgz
!pip install -q findspark


In [3]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.3.0-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("TitanicSparkLab5").getOrCreate()
print("Spark version:", spark.version)

Spark version: 3.3.0


In [4]:
from pyspark.sql.functions import lit

train_path = "/content/drive/MyDrive/iot/titanic/train.csv"
test_path  = "/content/drive/MyDrive/iot/titanic/test.csv"

train_df = spark.read.csv(train_path, header=True, inferSchema=True).na.drop()
test_df  = spark.read.csv(test_path,  header=True, inferSchema=True).na.drop()


# Add a dummy 'Survived' column to test_df
test_df = test_df.withColumn("Survived", lit(0))


# 2) Rename the "Survived" column to "label" so that Spark knows it's the target
train_df = train_df.withColumnRenamed("Survived", "label")
test_df  = test_df.withColumnRenamed("Survived", "label")

train_df.printSchema()
test_df.printSchema()




root
 |-- PassengerId: integer (nullable = true)
 |-- label: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)

root
 |-- PassengerId: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)
 |-- label: integer (nullable = false)



In [5]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Stage 1: Convert "Sex" -> "SexIndexed"
sex_indexer = StringIndexer(inputCol="Sex", outputCol="SexIndexed")

# Stage 2: Convert "Embarked" -> "EmbarkedIndexed"
embark_indexer = StringIndexer(inputCol="Embarked", outputCol="EmbarkedIndexed")

# Stage 3: Assemble features
assembler = VectorAssembler(
    inputCols=["Pclass", "SexIndexed", "Age", "Fare", "EmbarkedIndexed"],
    outputCol="features"
)

# Stage 4: Logistic Regression Model (using labelCol="label")
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# Combine stages into a pipeline
pipeline = Pipeline(stages=[sex_indexer, embark_indexer, assembler, lr])


In [6]:
pipeline_model = pipeline.fit(train_df)


In [7]:
predictions = pipeline_model.transform(test_df)
predictions.show(5)


+-----------+------+--------------------+------+----+-----+-----+-----------+-------+---------------+--------+-----+----------+---------------+--------------------+--------------------+--------------------+----------+
|PassengerId|Pclass|                Name|   Sex| Age|SibSp|Parch|     Ticket|   Fare|          Cabin|Embarked|label|SexIndexed|EmbarkedIndexed|            features|       rawPrediction|         probability|prediction|
+-----------+------+--------------------+------+----+-----+-----+-----------+-------+---------------+--------+-----+----------+---------------+--------------------+--------------------+--------------------+----------+
|        904|     1|Snyder, Mrs. John...|female|23.0|    1|    0|      21228|82.2667|            B45|       S|    0|       1.0|            0.0|[1.0,1.0,23.0,82....|[-3.1757090999752...|[0.04009013482227...|       1.0|
|        906|     1|Chaffee, Mrs. Her...|female|47.0|    1|    0|W.E.P. 5734| 61.175|            E31|       S|    0|       1.0| 

In [8]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print("Pipeline (LR) Test Accuracy =", accuracy)


Pipeline (LR) Test Accuracy = 0.3333333333333333
